<a href="https://colab.research.google.com/github/yurisalesc/minicurso-python-para-engenharia-e-ia/blob/main/minicurso_python_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Gerando dados sintéticos para o curso
np.random.seed(42)
n_rows = 500

data = {
    'ID_Pedido': range(1001, 1001 + n_rows),
    'Data': [datetime(2023, 1, 1) + timedelta(days=np.random.randint(0, 365)) for _ in range(n_rows)],
    'Produto': np.random.choice(['Laptop', 'Mouse', 'Monitor', 'Teclado', 'Headset'], n_rows),
    'Categoria': np.random.choice(['Eletrônicos', 'Acessórios'], n_rows),
    'Quantidade': np.random.randint(1, 5, n_rows),
    'Preco_Unitario': np.random.uniform(50, 5000, n_rows).round(2),
    'Regiao': np.random.choice(['Norte', 'Sul', 'Leste', 'Oeste'], n_rows),
    'Satisfacao_Cliente': np.random.randint(1, 6, n_rows)
}

df_original = pd.DataFrame(data)
df_original['Faturamento'] = df_original['Quantidade'] * df_original['Preco_Unitario']

# Salvar para simular leitura de arquivo
df_original.to_csv('vendas_curso.csv', index=False)
print('Arquivo vendas_curso.csv gerado com sucesso!')

### 1. Introdução à Análise de Dados com Pandas

**O que é o Pandas?**
O Pandas é a biblioteca padrão ouro para manipulação de dados tabulares (como planilhas Excel) em Python. Em projetos de IA, ele é essencial para a fase de **Data Wrangling** (limpeza e transformação).

**Principais Estruturas:**
- **Series:** Matriz unidimensional (como uma coluna única).
- **DataFrame:** Estrutura bidimensional (tabela completa com linhas e colunas).

**Inspeção Inicial:**
- `info()`: Mostra tipos de dados e valores nulos.
- `describe()`: Resume a estatística descritiva (média, desvio padrão, quartis).
- `head()`: Visualiza as primeiras amostras para entender a 'cara' dos dados.

In [ ]:
# Leitura e Inspeção
df = pd.read_csv('vendas_curso.csv')

print("--- Info Geral ---")
display(df.info())

print("\n--- Estatísticas Descritivas ---")
display(df.describe())

print("\n--- Primeiras Linhas ---")
display(df.head())

### 2. Filtragem e Agrupamentos (O coração da análise)

A análise exploratória busca responder perguntas de negócio. Para isso, usamos:

- **Filtros Condicionais:** Selecionar subconjuntos de dados (Ex: 'Apenas vendas acima de R$ 1000').
- **Groupby:** A técnica de 'Dividir-Aplicar-Combinar'. Agrupamos dados por uma categoria (ex: Região) e aplicamos uma função matemática (soma, média) para ver o comportamento macro dos dados.

In [ ]:
# 1. Filtro: Apenas vendas da região Sul com Satisfação < 3
baixa_satisfacao_sul = df[(df['Regiao'] == 'Sul') & (df['Satisfacao_Cliente'] < 3)]

# 2. Agrupamento: Faturamento total por Categoria e Produto
ranking_vendas = df.groupby(['Categoria', 'Produto'])['Faturamento'].sum().sort_values(ascending=False).reset_index()

display(ranking_vendas)

### 3. Visualização: Do Estático ao Interativo

Gráficos não são apenas para apresentações; são ferramentas de diagnóstico.

- **Matplotlib:** Ideal para gráficos estáticos em relatórios técnicos e publicações. É a base de quase todas as bibliotecas de plotagem em Python.
- **Bokeh:** Focado em interatividade Web. Permite que o usuário faça zoom, passe o mouse para ver valores (tooltips) e explore os dados de forma dinâmica. Essencial para dashboards onde o tomador de decisão precisa 'investigar' os pontos.

In [ ]:
import matplotlib.pyplot as plt
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool
import pandas as pd
output_notebook()

# --- 1. MATPLOTLIB: Mais Exemplos ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Histograma de Preços (Distribuição)
df['Preco_Unitario'].hist(bins=20, ax=ax1, color='skyblue', edgecolor='black')
ax1.set_title('Distribuição de Preços Unitários')
ax1.set_xlabel('Preço')
ax1.set_ylabel('Frequência')

# Gráfico de Barras: Faturamento por Região
df.groupby('Regiao')['Faturamento'].sum().sort_values().plot(kind='barh', ax=ax2, color='salmon')
ax2.set_title('Faturamento Total por Região')
ax2.set_xlabel('R$')

plt.tight_layout()
plt.show()

# --- 2. BOKEH: Interatividade Avançada ---
# Criando um subset para não sobrecarregar o gráfico
source = ColumnDataSource(df.sample(100))

tooltips = [
    ("Produto", "@Produto"),
    ("Faturamento", "@Faturamento{0,0.00}"),
    ("Região", "@Regiao"),
    ("Satisfação", "@Satisfacao_Cliente")
]

p = figure(title="Exploração Interativa: Preço vs Faturamento (Amostra)",
           x_axis_label='Preço Unitário',
           y_axis_label='Faturamento',
           tools="pan,wheel_zoom,box_zoom,reset,hover",
           tooltips=tooltips)

p.circle('Preco_Unitario', 'Faturamento', size=10, source=source,
         color="navy", alpha=0.6, hover_color="red")

show(p)

### 4. Conexão com Engenharia e Inteligência Artificial

Nesta fase, preparamos o terreno para algoritmos de ML. O objetivo é transformar dados que humanos entendem em matrizes matemáticas otimizadas.

**Conceitos Chave Detalhados:**

- **One-Hot Encoding:** Para categorias sem ordem (ex: Região), criamos colunas 0/1. Se usássemos 1, 2, 3, o modelo acharia que 'Sul' é maior que 'Norte', o que é um erro de viés.

- **Feature Scaling (Normalização):**
    - **Por que fazer?** Algoritmos baseados em distância (como KNN) ou gradiente (Redes Neurais) são sensíveis à escala. Se o 'Faturamento' está na casa dos 5.000 e a 'Quantidade' entre 1 e 5, o modelo dará um peso desproporcional ao faturamento.
    - **Min-Max Scaling:** Transforma os valores para um intervalo entre 0 e 1.

- **Temporalidade (Feature Engineering Temporal):**
    - Modelos não conseguem extrair padrões diretamente de objetos de data (Ex: 2023-12-25).
    - **Decomposição:** Extraímos o 'Mês' para detectar sazonalidade (ex: vendas de Natal) e o 'Dia da Semana' para entender comportamentos de fim de semana (ex: mais lazer, menos trabalho).

### Entendendo o One-Hot Encoding com Múltiplas Categorias

Quando usamos `pd.get_dummies`, transformamos uma coluna de texto com $N$ categorias em $N$ colunas binárias.

**Exemplo prático:**
Se temos a coluna `Regiao` com os valores: `['Norte', 'Sul', 'Leste']`.

O Pandas transformará em:
| Pedido | Reg_Norte | Reg_Sul | Reg_Leste |
| :--- | :---: | :---: | :---: |
| Pedido 1 (Norte) | **1** | 0 | 0 |
| Pedido 2 (Sul) | 0 | **1** | 0 |
| Pedido 3 (Leste) | 0 | 0 | **1** |

Isso evita o problema do **Label Encoding** (atribuir 1, 2, 3), onde um modelo de IA poderia interpretar erroneamente que Leste (3) vale três vezes mais que Norte (1).

In [ ]:
# Criando um exemplo rápido para visualizar o efeito em múltiplas regiões
df_exemplo = pd.DataFrame({'Regiao': ['Norte', 'Sul', 'Leste', 'Oeste', 'Sul']})

# Aplicando One-Hot Encoding
df_dummies = pd.get_dummies(df_exemplo, prefix='Reg')

print("DataFrame Original:")
display(df_exemplo)

print("\nApós get_dummies (One-Hot Encoding):")
display(df_dummies)

In [ ]:
# 1. One-Hot Encoding para Variáveis Categóricas
df_ml = pd.get_dummies(df, columns=['Regiao', 'Categoria'], prefix=['Reg', 'Cat'])

# 2. Engenharia de Atributos Temporais (Sazonalidade)
df_ml['Data'] = pd.to_datetime(df_ml['Data'])
df_ml['Mes'] = df_ml['Data'].dt.month
df_ml['Dia_Semana'] = df_ml['Data'].dt.dayofweek # 0=Segunda, 6=Domingo

# 3. Criação de Alvos (Targets) para Classificação
# Exemplo: O cliente teve uma experiência ruim? (Satisfação <= 2)
df_ml['Experiencia_Ruim'] = (df_ml['Satisfacao_Cliente'] <= 2).astype(int)

# 4. Normalização Simples (Min-Max Scaling manual para fins didáticos)
# Colocando o faturamento na escala [0, 1]
df_ml['Faturamento_Norm'] = (df_ml['Faturamento'] - df_ml['Faturamento'].min()) / (df_ml['Faturamento'].max() - df_ml['Faturamento'].min())

print("Base pronta para treinamento de modelos de IA:")
display(df_ml[['Produto', 'Mes', 'Dia_Semana', 'Faturamento_Norm', 'Reg_Sul', 'Experiencia_Ruim']].head())

### 4.1 Aplicando Machine Learning com Scikit-Learn

Agora que os dados estão preparados (numéricos e em escala), podemos treinar modelos. Vamos usar o **Random Forest (Floresta Aleatória)**, um dos algoritmos mais poderosos para dados tabulares.

**Conceitos:**
- **Train/Test Split:** Dividimos os dados em 80% para o modelo aprender e 20% para testarmos se ele realmente aprendeu ou apenas decorou (overfitting).
- **Regressão:** Prever um número contínuo (Ex: Qual será o faturamento?).
- **Classificação:** Prever uma categoria (Ex: Este cliente está insatisfeito? Sim/Não).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, accuracy_score

# Preparando X (recursos) e y (alvos)
# Vamos remover colunas que o modelo não consegue ler diretamente (strings e datas brutas)
features = df_ml.drop(columns=['ID_Pedido', 'Data', 'Produto', 'Categoria', 'Satisfacao_Cliente', 'Faturamento', 'Experiencia_Ruim', 'Faturamento_Norm'])

# --- CENÁRIO A: REGRESSÃO (Prever Faturamento) ---
y_reg = df_ml['Faturamento']
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(features, y_reg, test_size=0.2, random_ Harris_seed=42)

model_reg = RandomForestRegressor(n_estimators=100, random_state=42)
model_reg.fit(X_train_r, y_train_r)
preds_reg = model_reg.predict(X_test_r)

# --- CENÁRIO B: CLASSIFICAÇÃO (Prever Experiência Ruim) ---
y_clf = df_ml['Experiencia_Ruim']
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(features, y_clf, test_size=0.2, random_state=42)

model_clf = RandomForestClassifier(random_state=42)
model_clf.fit(X_train_c, y_train_c)
preds_clf = model_clf.predict(X_test_c)

print(f"Erro Médio na Previsão de Faturamento (MAE): R$ {mean_absolute_error(y_test_r, preds_reg):.2f}")
print(f"Acurácia na Previsão de Insatisfação: {accuracy_score(y_test_c, preds_clf)*100:.2f}%")

### Entendendo a 'Caixa Preta': Importância das Variáveis

Uma das grandes vantagens do Random Forest é que ele nos diz quais colunas foram mais importantes para a tomada de decisão do modelo. Isso é o que chamamos de **Feature Importance**.

In [ ]:
importances = pd.Series(model_reg.feature_importances_, index=features.columns)
importances.sort_values().plot(kind='barh', title='Quais fatores mais influenciam o Faturamento?')
plt.show()

### 5. Atividade Prática Final
**Desafio:**
1. Carregue o CSV.
2. Crie uma nova coluna chamada `Ticket_Medio` (Faturamento / Quantidade).
3. Agrupe por `Regiao` e calcule a média de satisfação.
4. Gere um gráfico de barras mostrando o faturamento por Região.

### Resolução da Atividade Prática Final

Aqui aplicamos os 4 passos solicitados no desafio para consolidar o aprendizado do curso.

In [ ]:
# 1. Carregar o CSV
df_final = pd.read_csv('vendas_curso.csv')

# 2. Criar a coluna Ticket_Medio
df_final['Ticket_Medio'] = df_final['Faturamento'] / df_final['Quantidade']

# 3. Agrupar por Regiao e calcular a média de satisfação
satisfacao_media = df_final.groupby('Regiao')['Satisfacao_Cliente'].mean().sort_values(ascending=False)

print("Satisfação Média por Região:")
display(satisfacao_media)

# 4. Gráfico de barras: Faturamento por Região
plt.figure(figsize=(10, 6))
faturamento_regiao = df_final.groupby('Regiao')['Faturamento'].sum().sort_values()
faturamento_regiao.plot(kind='bar', color='teal')

plt.title('Faturamento Total por Região')
plt.xlabel('Região')
plt.ylabel('Faturamento Total (R$)')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()